# Linear Regression Modeling for Stock Price Prediction

## Project Overview

This project uses **Ordinary Least Squares (OLS) Linear Regression** to model the next-day price of **Apple (AAPL)**.

The model uses historical information from:
- AAPL
- S&P 500 (^GSPC)

The project includes:

- Historical data collection using Yahoo Finance
- Feature engineering using lagged prices
- 5-day moving averages during the initial model exploration
- OLS linear regression
- Actual vs. predicted price comparison
- Linear regression assumption checks
- Out-of-sample testing using 2025 data
- MSE and RMSE evaluation

### Objective

The objective is to understand whether a simple linear regression model can capture short-term movements in AAPL's stock price.

**Data source:** Yahoo Finance via `yfinance`.

**Note:** This is an educational project. Stock-price prediction is difficult and the model should not be interpreted as investment advice.


In [ ]:
# Quant Project 2: Linear Regression Modeling for predicting the Stock P
# Step 1: Download the data from Yahoo Finance
# Step 2: Some Feature Engineering (to build new features) - Technical In
# Step 3: Run Linear Regression Model
# Step 4: Check how the model is performed (Actual vs Predicted)
# Step 5: Test for all the assumptions
# Step 6: Check the prediction


In [ ]:
# Import required packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import statsmodels.api as sm


In [ ]:
# Step 1: Download the data from Yahoo Finance
tickers = ['AAPL', 'AMZN', 'MSFT', 'QQQ', '^GSPC']
df = yf.download(tickers, start = '2020-01-01', end = '2024-12-31')['Close']
df

In [ ]:
# Step 2: Perform Feature Engineering

# Considering Yesterday's Value
df['AAPL(t-1)'] = df['AAPL'].shift(1)
df['AMZN(t-1)'] = df['AMZN'].shift(1)
df['MSFT(t-1)'] = df['MSFT'].shift(1)
df['QQQ(t-1)'] = df['QQQ'].shift(1)
df['^GSPC(t-1)'] = df['^GSPC'].shift(1)
# Moving Avg (MA): Technical Indicator 
df['AAPL_MA_5'] = df['AAPL'].rolling(window=5).mean()
df['AMZN_MA_5'] = df['AMZN'].rolling(window=5).mean()
df['MSFT_MA_5'] = df['MSFT'].rolling(window=5).mean()
df['QQQ_MA_5'] = df['QQQ'].rolling(window=5).mean()
df['^GSPC_MA_5'] = df['^GSPC'].rolling(window=5).mean()
# Set Y Variable - Next day
df['Target'] = df['AAPL'].shift(-1)
df = df.dropna()


In [ ]:
# Step 3: Fit a Linear Regression Model
# Set X and Y variable for Linear Regression Model - Ordinary Least Squares (OLS)
import statsmodels.api as sm
X = df[['AAPL(t-1)', 'AMZN(t-1)',
 'MSFT(t-1)', 'QQQ(t-1)', '^GSPC(t-1)', 'AAPL_MA_5', 'AMZN_MA_5',
 'MSFT_MA_5', 'QQQ_MA_5', '^GSPC_MA_5']]
Y = df['Target']
X_const = sm.add_constant(X) # Intercept Term
# Train the Model
model = sm.OLS(Y, X_const).fit()
# Summary
model.summary()


In [ ]:
# Set X and Y variable for Linear Regression Model - Ordinary Least Squa
import statsmodels.api as sm
X = df[['AAPL(t-1)', '^GSPC(t-1)']] # Dropping AAPL_MA_5
Y = df['Target']
X_const = sm.add_constant(X) # Intercept Term
# Train the Model
model = sm.OLS(Y, X_const).fit()
# Summary
model.summary()

In [ ]:
df_train_predict = pd.DataFrame()
df_train_predict['Actual'] = df['Target']
df_train_predict['Predicted'] = model.predict(X_const)
df_train_predict


In [ ]:
# Plot between Actual vs Predicted Value
# Initialize the plot size
plt.figure(figsize=(14, 6))

# Plot actual vs predicted values with completed labels and quotes
plt.plot(df_train_predict.index, df_train_predict['Actual'], label='Actual')
plt.plot(df_train_predict.index, df_train_predict['Predicted'], label='Predicted')

# Add titles, labels, and display the legend
plt.title("Actual vs Prediction for AAPL Stock")
plt.xlabel("Date")
plt.ylabel("Stock Price")
plt.legend()  # Added to display the 'Actual' and 'Predicted' labels

# Render the plot
plt.show()


In [ ]:
# Assumptions of Linear Regression:
# 1) Linearity between dependent and independent variable - Met
import seaborn as sns
df = df[['AAPL', 'AAPL(t-1)', '^GSPC(t-1)']]
sns.pairplot(df)
# AAPL & AAPL(t-1) has linear relationship
# AAPL & S&P500 has linear relationship
# AAPL & AAPL_MA_5 has linear relationship

In [ ]:
# 2) Homoscedasticity: Fitting Residual with the predicted value
residual = model.resid # Actual - Predicted
fitted = model.fittedvalues # Predicted Y Value
plt.figure(figsize = (8,5))
sns.scatterplot(x = fitted, y = residual)
plt.axhline(0, color = 'red', linestyle = '--')
plt.title('Test for Homoscedasticity')
plt.xlabel('Fitted Values')
plt.ylabel('Residuals')
plt.show()
# Since it's a tube like structure => It is homoscedastic -> Assumption i
# If it was funnel like structure => It is heteroscedastic


In [ ]:
# 3) Multicollinearity => VIF (Variance Inflation Factor) => For independent variables
# Rule of thumb for VIF
# VIF < 1 => No Multicollinearity
# VIF < 10 => Moderate Multicollinearity
# VIF > 10 => Strong Multicollinearity
# VIF = Condition is Met

from statsmodels.stats.outliers_influence import variance_inflation_factor

# Initialize an empty DataFrame for VIF results
vif = pd.DataFrame()
vif['Features'] = X_const.columns

# Calculate VIF for each feature using a list comprehension
vif['VIF'] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]

# Drop the constant/intercept row (index 0) if you only want to see independent variables
vif = vif[1:]
vif


In [ ]:
# 4) Assumption: Normality of Residual => 1) Visual Test (Histogram) or Q-Q Plot
plt.figure(figsize = (6,4))
plt.hist(residual, bins = 30)
plt.title("Normality of Residuals")
plt.show()


In [ ]:
# Q-Q plot for checking residual normality
sm.qqplot(residual, line='45', fit=True)
plt.title("Q-Q Plot of Residuals")
plt.show()


In [ ]:
# Test 5: Auto correlation of Residual: Durbin Watson Test
from statsmodels.stats.stattools import durbin_watson
dw = durbin_watson(residual)
dw # p value
# p value < 0.05 => Autocorrelation b/w residual is there
# p value > 0.05 => Autocorrelation b/w residual is not there
# Our 5th condition is met

In [ ]:
# All the 5 conditions of Linear Regression Modela are met
# Linearity
# Homoscedasticity
# VIF
# Normality of Residuals
# Auto correlation of Residual


## Out-of-Sample Test: 2025

The model is trained using historical data from 2020–2024 and then tested on AAPL data from January–March 2025.

This gives a simple out-of-sample check rather than evaluating the model only on the data used for training.


In [ ]:
# Step 1: Download the data from Yahoo Finance
tickers = ['AAPL', '^GSPC']
df = yf.download(tickers, start = '2025-01-01', end = '2025-03-31')['Close']
df.head()

In [ ]:
# Step 2: Data Eng
df['AAPL(t-1)'] = df['AAPL'].shift(1)
df['^GSPC(t-1)'] = df['^GSPC'].shift(1)
df = df.dropna()
df.head()

In [ ]:
X_test = df[['AAPL(t-1)', '^GSPC(t-1)']]
X_test = sm.add_constant(X_test)
df_result = pd.DataFrame()
df_result['Actual'] = df['AAPL']
df_result['Predicted'] = model.predict(X_test)
df_result.head()


In [ ]:
# Actual vs Predicted AAPL price during the 2025 test period
plt.figure(figsize=(14, 6))
plt.plot(df_result.index, df_result["Actual"], label="Actual")
plt.plot(df_result.index, df_result["Predicted"], label="Predicted")
plt.title("Actual vs Predicted AAPL Stock Price (2025)")
plt.xlabel("Date")
plt.ylabel("Stock Price")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Model error metrics
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(df_result["Actual"], df_result["Predicted"])
rmse = np.sqrt(mse)

print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")


In [ ]:
# Conclusion: It's a decent Model but not 100% Accurate
# Lesson: Stock data in general have lot of non linearities
# It's extremely tough to use simple linear regression model just to capature the stock price movement

## Notes on the Model

- **OLS:** estimates the relationship between the target price and the selected explanatory variables.
- **Lagged price:** `AAPL(t-1)` represents the previous trading day's AAPL price.
- **S&P 500 lagged price:** `^GSPC(t-1)` represents the previous trading day's S&P 500 price.
- **MSE:** measures the average squared prediction error.
- **RMSE:** measures prediction error in the same units as the stock price.

The model is deliberately simple. Its purpose is to demonstrate the application of linear regression to financial data rather than to build a production trading system.


## Conclusion

The results show that a simple linear regression model can capture some relationship between historical market variables and AAPL's next-day price, but it does not perfectly predict stock-price movements.

Stock prices are affected by many factors and may exhibit nonlinear behavior. Therefore, this project is best viewed as an introduction to applying statistical regression methods to financial data.

### Limitations

- The model uses a small number of explanatory variables.
- Stock prices are not necessarily well described by a linear relationship.
- The model does not include transaction costs or trading signals.
- Historical relationships may change over time.
- The 2025 test period is limited to January–March.
